In [1]:
import matplotlib.pyplot as plt
%matplotlib inline

#@title import matplotlib and set defaults
from matplotlib import rcParams 
from matplotlib import pyplot as plt
rcParams['figure.figsize'] = [20, 4]
rcParams['font.size'] =15
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['figure.autolayout'] = True

In [2]:
#@title groupings of brain regions
regions = ["vis ctx", "thal", "hipp", "other ctx", "midbrain",  "basal ganglia", "subplate"]
brain_groups = [["VISa", "VISam", "VISl", "VISp", "VISpm", "VISrl"], # visual cortex
                ["CL", "LD", "LGd", "LH", "LP", "MD", "MG", "PO", "POL", "PT", "RT", "SPF", "TH", "VAL", "VPL", "VPM"], # thalamus
                ["CA", "CA1", "CA2", "CA3", "DG", "SUB", "POST"], # hippocampal
                ["ACA", "AUD", "COA", "DP", "ILA", "MOp", "MOs", "OLF", "ORB", "ORBm", "PIR", "PL", "SSp", "SSs", "RSP"," TT"], # non-visual cortex
                ["APN", "IC", "MB", "MRN", "NB", "PAG", "RN", "SCs", "SCm", "SCig", "SCsg", "ZI"], # midbrain
                ["ACB", "CP", "GPe", "LS", "LSc", "LSr", "MS", "OT", "SNr", "SI"], # basal ganglia 
                ["BLA", "BMA", "EP", "EPd", "MEA"] # cortical subplate
               ]

In [10]:
import glob, os, sys
import numpy as np
from scipy.stats import zscore, spearmanr, mannwhitneyu, pearsonr
import time
import imp
import steinmetz_loader
from scipy.sparse import csr_matrix

imp.reload(steinmetz_loader)

dataroot = './steinmetz_data/raw_sessions/'
print(f"dataroot is: {dataroot}") # <-- ADD THIS LINE FOR DEBUGGING


fdir = glob.glob(os.path.join(dataroot, "*", ""))
print(f"Found directories (fdir): {fdir}") # <-- ADD THIS LINE FOR DEBUGGING
if not fdir:
    raise ValueError(f"No session directories found in {dataroot}. Please check the path and that session folders (e.g. Cori_2016-12-14) exist directly inside it.")


dt = 1/100
dT = 2.5
T0 = .5

dat = []

for idir in range(len(fdir)):
    print(f"Processing directory: {fdir[idir]}") # DEBUG
    # good cells and brain regions
    good_cells, brain_region, br = steinmetz_loader.get_good_cells(fdir[idir])
    
    # event types
    response, vis_right, vis_left, feedback_type = steinmetz_loader.get_event_types(fdir[idir])
    
    # event timing
    response_times, visual_times, rsp, gocue, feedback_time = steinmetz_loader.get_event_times(fdir[idir])    

    # get passive trials
    vis_times_p, vis_right_p, vis_left_p = steinmetz_loader.get_passive(fdir[idir])
    visual_times = np.vstack((visual_times, vis_times_p))
    vis_right = np.hstack((vis_right, vis_right_p))
    vis_left  = np.hstack((vis_left, vis_left_p))
    
    # wheel traces
    stimes, sclust    = steinmetz_loader.get_spikes(fdir[idir])
    
    # only care about spikes during trials
    wheel, wheel_times = steinmetz_loader.get_wheel(fdir[idir])
    
    # load the pupil
    pup, xy, pup_times = steinmetz_loader.get_pup(fdir[idir])

    # load the LFP
    L, ba_lfp = steinmetz_loader.get_LFP(fdir[idir], br, visual_times-T0, dT, dt, T0)
    
    # trials loader
    S  = steinmetz_loader.psth(stimes, sclust,   visual_times-T0, dT, dt)
    
    # wheel trials
    W = steinmetz_loader.wpsth(wheel, wheel_times,   visual_times-T0, dT, dt)
    
    # pupil loader
    P = steinmetz_loader.ppsth(pup, pup_times,   visual_times-T0, dT, dt)
    
    # add spike waveform information
    twav, w, u = steinmetz_loader.get_waves(fdir[idir])
    
    
    good_cells = good_cells * (np.mean(S, axis=(1,2))>0)
    S  = S[good_cells].astype('int8') 
   
    dat.append({})
    ntrials = len(dat[idir]['response'])
    
    dat[idir]['brain_area'] = brain_region[good_cells]
    dat[idir]['spks'] = S[:, :ntrials, :]
    dat[idir]['wheel'] = W[np.newaxis, :ntrials, :]
    dat[idir]['pupil'] = P[:, :ntrials, :]
    dat[idir]['response'] = response
    dat[idir]['contrast_right'] = vis_right[:ntrials]
    dat[idir]['contrast_left'] = vis_left[:ntrials]
    dat[idir]['response_time'] = rsp
    dat[idir]['feedback_time'] = feedback_time
    dat[idir]['feedback_type'] = feedback_type  
    dat[idir]['gocue'] = gocue
    dat[idir]['mouse_name'] = fdir[idir].split('\\')[1].split('_')[0]
    dat[idir]['date_exp'] = fdir[idir].split('\\')[1].split('_')[1]
    dat[idir]['trough_to_peak'] = twav[good_cells].flatten()
    dat[idir]['waveform_w'] = w[good_cells].astype('float32')
    dat[idir]['waveform_u'] = u[good_cells].astype('float32')
    dat[idir]['bin_size'] = dt
    dat[idir]['stim_onset'] = T0
    
    dat[idir]['spks_passive'] = S[:, ntrials:, :]
    dat[idir]['wheel_passive'] = W[np.newaxis, ntrials:, :]
    dat[idir]['pupil_passive'] = P[:, ntrials:, :]
    dat[idir]['lfp_passive'] = L[:, ntrials:, :]
    dat[idir]['contrast_right_passive'] = vis_right[ntrials:]
    dat[idir]['contrast_left_passive'] = vis_left[ntrials:]
        
    # add LFP
    L, ba_lfp = steinmetz_loader.get_LFP(fdir[idir], br, visual_times-T0, dT, dt, T0)
    dat[idir]['lfp'] = L[:, :ntrials, :]
    dat[idir]['lfp_passive'] = L[:, ntrials:, :]
    dat[idir]['brain_area_lfp'] = ba_lfp
  
    #S  = np.reshape(S[good_cells], (np.sum(good_cells), -1))
    #sall.append(csr_matrix(S))
    

dataroot is: ./steinmetz_data/raw_sessions/
Found directories (fdir): ['./steinmetz_data/raw_sessions/Cori_2016-12-14/', './steinmetz_data/raw_sessions/Cori_2016-12-17/', './steinmetz_data/raw_sessions/Cori_2016-12-18/', './steinmetz_data/raw_sessions/Forssmann_2017-11-01/', './steinmetz_data/raw_sessions/Forssmann_2017-11-02/', './steinmetz_data/raw_sessions/Forssmann_2017-11-04/', './steinmetz_data/raw_sessions/Forssmann_2017-11-05/', './steinmetz_data/raw_sessions/Hench_2017-06-15/', './steinmetz_data/raw_sessions/Hench_2017-06-16/', './steinmetz_data/raw_sessions/Hench_2017-06-17/', './steinmetz_data/raw_sessions/Hench_2017-06-18/', './steinmetz_data/raw_sessions/Lederberg_2017-12-05/', './steinmetz_data/raw_sessions/Lederberg_2017-12-06/', './steinmetz_data/raw_sessions/Lederberg_2017-12-07/', './steinmetz_data/raw_sessions/Lederberg_2017-12-08/', './steinmetz_data/raw_sessions/Lederberg_2017-12-09/', './steinmetz_data/raw_sessions/Lederberg_2017-12-10/', './steinmetz_data/raw_ses

FileNotFoundError: [Errno 2] No such file or directory: './steinmetz_data/raw_sessions/Cori_2016-12-14/Cori_2016-12-14_M2_g0_t0.imec.lf.bin'

In [ ]:


# # IMPORTANT: Change range(1) to range(len(fdir)) to process all sessions once fdir is populated
# # Or range(N) for a subset, e.g. range(3) to test with the first 3 sessions
# for idir in range(len(fdir)): # MODIFIED TO PROCESS ALL FOUND DIRECTORIES (or test with a small number first)
#     print(f"Processing directory: {fdir[idir]}") # DEBUG
#     # good cells and brain regions
#     good_cells, brain_region, br = steinmetz_loader.get_good_cells(fdir[idir])
    
#     # event types
#     response, vis_right, vis_left, feedback_type = steinmetz_loader.get_event_types(fdir[idir])
    
#     # event timing
#     response_times, visual_times, rsp, gocue, feedback_time = steinmetz_loader.get_event_times(fdir[idir])    

#     # get passive trials
#     vis_times_p, vis_right_p, vis_left_p = steinmetz_loader.get_passive(fdir[idir])
#     # ... (rest of the cell remains the same)

In [ ]:
dat[0].keys()

In [ ]:
current_dir = os.getcwd()

In [ ]:
np.savez_compressed(f'{current_dir}/steinmetz-data/steinmetz_part0.npz', dat = dat[:13])
np.savez_compressed(f'{current_dir}/steinmetz_part1.npz', dat = dat[13:26])
np.savez_compressed(f'{current_dir}/steinmetz_part2.npz', dat = dat[26:])